In [ ]:
!pip install pymupdf transformers accelerate bitsandbytes torchvision pillow

In [ ]:
!pip install typhoon-ocr -U
!pip install vllm
!vllm serve scb10x/typhoon-ocr1.5-2b --max-model-len 49152 --served-model-name typhoon-ocr-1-5

In [ ]:
!curl http://localhost:8000/v1/models

In [ ]:
import os, fitz, re
from collections import Counter
from typhoon_ocr import ocr_document
from pythainlp.util import normalize
from pythainlp.tokenize import word_tokenize

# รายชื่อไฟล์ PDF ที่ต้องการประมวลผล
pdf_files = ["Sample1pdf.pdf", "TOR-เสาไฟกินรีราชเทวะ.PDF"]

# โฟลเดอร์สำหรับเก็บรูปภาพที่แปลงจาก PDF
output_folder = "pymupdf_pdf_to_images"

# ตั้งค่า API สำหรับ OCR (Typhoon OCR)
OCR_BASE_URL = "http://localhost:11434/v1"
OCR_API_KEY = "ollama"
OCR_MODEL_NAME = "scb10x/typhoon-ocr1.5-3b"

# สร้างโฟลเดอร์ถ้ายังไม่มี
os.makedirs(output_folder, exist_ok=True)

# PDF → Images
def pdf_to_images(pdf_path):
    images = []
    try:
        # เปิดไฟล์ PDF
        doc = fitz.open(pdf_path)
        for page_num in range(len(doc)):
            pix = doc.load_page(page_num).get_pixmap(matrix=fitz.Matrix(300/72, 300/72))
            img_path = os.path.join(
                output_folder,
                f"{os.path.splitext(os.path.basename(pdf_path))[0]}_page_{page_num+1}.png"
            )
            pix.save(img_path)
            images.append(img_path)
        doc.close()
    except Exception as e:
        print(f"Error converting {pdf_path}: {e}")
    return images

# OCR → Raw Text + Confidence Filtering
def run_ocr_with_confidence(images, threshold=0.7):
    pages_text = []
    for idx, img_path in enumerate(images, start=1):
        try:
            # เรียกใช้งาน OCR API
            result = ocr_document(
                img_path,
                model=OCR_MODEL_NAME,
                figure_language="Thai",
                task_type="v1.5",
                base_url=OCR_BASE_URL,
                api_key=OCR_API_KEY,
                return_confidence=True
            )

            # กรณี OCR คืนค่าเป็น string ธรรมดา
            if isinstance(result, str):
                pages_text.append(result)

            # กรณี OCR คืนค่าเป็น list ของ dict
            elif isinstance(result, list):
                filtered_text = []
                for item in result:
                    if isinstance(item, dict):
                        if item.get("confidence", 1.0) >= threshold:
                            filtered_text.append(item.get("text", ""))
                        else: # ถ้า confidence ต่ำ → ทำเครื่องหมาย LowConf
                            filtered_text.append(f"(LowConf:{item.get('text', '')})")
                    else:
                        filtered_text.append(str(item))
                pages_text.append(" ".join(filtered_text))
            # กรณี OCR คืนค่าเป็น dict เดี่ยว
            elif isinstance(result, dict):
                if result.get("confidence", 1.0) >= threshold:
                    pages_text.append(result.get("text", ""))
                else:
                    pages_text.append(f"(LowConf:{result.get('text', '')})")
            # กรณีอื่น ๆ
            else:
                pages_text.append(str(result))

        except Exception as e:
            pages_text.append(f"*(OCR Error at page {idx})*")
    return pages_text

# Detect Header/Footer + RegEx Cleaning
def clean_pages(pages_text, threshold=0.8):
    if not pages_text:
        return []

    common_headers = set()
    common_footers = set()

    # จะคำนวณหา Header/Footer ร่วมก็ต่อเมื่อมีเอกสารตั้งแต่ 3 หน้าขึ้นไปเท่านั้น
    if len(pages_text) >= 3:
        headers = [p.splitlines()[0] for p in pages_text if p.splitlines()]
        footers = [p.splitlines()[-1] for p in pages_text if p.splitlines()]
        header_counts = Counter(headers)
        footer_counts = Counter(footers)

        common_headers = {h for h, c in header_counts.items() if c/len(pages_text) >= threshold}
        common_footers = {f for f, c in footer_counts.items() if c/len(pages_text) >= threshold}

    cleaned_pages = []
    for page in pages_text:
        lines = page.splitlines()
        # ลบ header/footer ที่เจอซ้ำ
        if lines and lines[0] in common_headers: lines = lines[1:]
        if lines and lines[-1] in common_footers: lines = lines[:-1]
        text = "\n".join(lines)

        # Cleaning ด้วย regex
        text = re.sub(r'หน้า\s*\d+|Page\s*\d+', '', text) # ลบเลขหน้า
        text = re.sub(r'(\S)\n(\S)', r'\1\2', text) # ลบ line break ที่ไม่จำเป็น
        text = text.replace('\n', ' ')
        text = re.sub(r'\s+', ' ', text).strip()

        cleaned_pages.append(text)
    return cleaned_pages

# PyThaiNLP Cleaning
def pythai_clean(text):
    normalized = normalize(text) # จัดการสระซ้อนและวรรณยุกต์จม
    tokens = word_tokenize(normalized, engine="newmm") # ตัดคำภาษาไทย
    return " ".join(tokens)

# MAIN PIPELINE
all_documents = {}

for pdf in pdf_files:
    print(f"=== Processing {pdf} ===")
    images = pdf_to_images(pdf)

    if not images:
        print(f"Skip {pdf}: ไม่สามารถแปลงไฟล์เป็นรูปภาพได้")
        continue

    raw_text = run_ocr_with_confidence(images)
    cleaned_text = clean_pages(raw_text)

    final_pages = [pythai_clean(t) for t in cleaned_text]

    # เก็บผลลัพธ์ลง Dictionary
    all_documents[pdf] = final_pages

print("\nเสร็จสิ้น ได้ผลลัพธ์เป็น list[str] แยกตามเอกสารเรียบร้อย")

=== Processing Sample1pdf.pdf ===
=== Processing TOR-เสาไฟกินรีราชเทวะ.PDF ===

เสร็จสิ้น ได้ผลลัพธ์เป็น list[str] แยกตามเอกสารเรียบร้อย


แยกเก็บเป็นไฟล์เอกสาร

In [2]:
for pdf, pages in all_documents.items():
    output_filename = f"{os.path.splitext(pdf)[0]}_cleaned.md"
    with open(output_filename, "w", encoding="utf-8") as f:
        for idx, page in enumerate(pages, start=1):
            f.write(f"# {pdf} - Page {idx}\n{page}\n\n")
    print(f"บันทึกไฟล์ {output_filename} เรียบร้อย")


บันทึกไฟล์ Sample1pdf_cleaned.md เรียบร้อย
บันทึกไฟล์ TOR-เสาไฟกินรีราชเทวะ_cleaned.md เรียบร้อย
